# Synthetic benchmark: sklearn SVM with RFE reference

This standalone extension selects exactly 20 features with sklearn `RFE` and a
linear SVC, then tunes an RBF SVC on those selected features. Selection,
tuning, and scaling use training data only; performance is measured on the
same fixed blind set as the MiSTIC benchmark.

In [1]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.feature_selection import RFE
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Make the notebook work from either the repository root or validation.
repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "mistic" / "svmSet.py").exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mistic import combined_rank, cvSet, kernelWrapper, paramSet, score_svc, svmSet

sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)

Matplotlib is building the font cache; this may take a moment.


In [2]:
N_SAMPLES = 500
N_FEATURES = 100
N_INFORMATIVE = 10
N_REDUNDANT = 10
SIGNAL_FEATURES = set(range(N_INFORMATIVE + N_REDUNDANT))

X_values, y_values = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_INFORMATIVE,
    n_redundant=N_REDUNDANT,
    n_repeated=0,
    n_classes=2,
    weights=[0.55, 0.45],
    class_sep=1.0,
    flip_y=0.03,
    shuffle=False,
    random_state=2026,
)
feature_names = [
    *(f"informative_{i:02d}" for i in range(N_INFORMATIVE)),
    *(f"redundant_{i:02d}" for i in range(N_REDUNDANT)),
    *(f"noise_{i:02d}" for i in range(N_FEATURES - N_INFORMATIVE - N_REDUNDANT)),
]
X = pd.DataFrame(X_values, columns=feature_names)
y = pd.Series(y_values, name="class")

BASE_MODEL_COUNTS = [1, 3, 5, 10, 20]
MODEL_COUNTS = [*BASE_MODEL_COUNTS, 30]
BLIND_SET_SEED = 42
INNER_SEEDS = list(range(3))
TEST_SIZE = 0.25
INNER_VALIDATION_SIZE = 0.20
RANK_WEIGHT = 0.75
SELECTION_STRATEGIES = ["backward"]

# A compact grid keeps the full selection-by-size experiment tractable.
C_VALUES = [0.25, 1.0, 4.0]
GAMMA_VALUES = [2.0 ** exponent for exponent in (-9, -7, -5)]

print(f"Samples: {len(X)}, features: {X.shape[1]}")
print(y.value_counts().sort_index())

Samples: 500, features: 100
class
0    275
1    225
Name: count, dtype: int64


In [3]:
def metric_row(y_true, predictions, decision_values):
    # Metrics computed only from the untouched blind-set observations.
    return {
        "roc_auc": roc_auc_score(y_true, decision_values),
        "f1": f1_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "accuracy": accuracy_score(y_true, predictions),
    }


def fit_sklearn_pipeline(X_train, y_train, seed):
    pipeline = Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(kernel="rbf", class_weight="balanced")),
    ])
    search = GridSearchCV(
        pipeline,
        param_grid={"svc__C": C_VALUES, "svc__gamma": GAMMA_VALUES},
        scoring="roc_auc",
        cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=seed),
        n_jobs=-1,
        refit=True,
    )
    return search.fit(X_train, y_train)


def fit_svm_set(X_train, y_train, num_models, seed, selection_strategy):
    # Fit the scaler on this outer-training split only.
    scaler = StandardScaler().fit(X_train)
    X_scaled = scaler.transform(X_train)

    splits = cvSet(X_scaled, np.asarray(y_train))
    splits.classification(
        num_sets=num_models,
        validation_size=INNER_VALIDATION_SIZE,
        random_seed=seed,
    )

    ensemble = svmSet(
        SVC(kernel="precomputed", class_weight="balanced"),
        splits,
        score_method=score_svc(weight=1.0).score,  # tune for ROC AUC
        kernel=kernelWrapper(type="rbf"),
        separate_feature_sets=True,
        separate_parameters=True,
    )
    parameter_grid = [
        paramSet(model={"C": cost}, kernel={"gamma": gamma})
        for cost in C_VALUES
        for gamma in GAMMA_VALUES
    ]
    selection_options = dict(
        parameter_grid=parameter_grid,
        feature_ranker=combined_rank(weight=RANK_WEIGHT).compute,
        set_for_rank="sample",
    )
    if selection_strategy != "backward":
        raise ValueError(f"unknown selection strategy: {selection_strategy}")
    ensemble.greedy_backward_selection(
        reduction_factor=0.1,
        tune_models_each_step=False,
        **selection_options,
    )
    return scaler, ensemble

In [4]:
RFE_FEATURE_COUNT = 20
rfe_rows = []
X_train, X_blind, y_train, y_blind = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=BLIND_SET_SEED,
)

for seed in INNER_SEEDS:
    selector_scaler = StandardScaler().fit(X_train)
    X_train_scaled = selector_scaler.transform(X_train)
    selector = RFE(
        estimator=SVC(kernel="linear", C=1.0, class_weight="balanced"),
        n_features_to_select=RFE_FEATURE_COUNT,
        step=0.1,
    ).fit(X_train_scaled, y_train)
    selected_features = np.flatnonzero(selector.support_)

    model = fit_sklearn_pipeline(
        X_train.iloc[:, selected_features], y_train, seed
    )
    selected_signal = len(set(selected_features).intersection(SIGNAL_FEATURES))
    rfe_rows.append({
        "seed": seed,
        "method": "sklearn SVM + RFE",
        "selection_strategy": "RFE",
        "num_models": 1,
        "num_unified_features": len(selected_features),
        "signal_recall": selected_signal / len(SIGNAL_FEATURES),
        "noise_fraction": 1 - selected_signal / len(selected_features),
        "member_disagreement": 0.0,
        **metric_row(
            y_blind,
            model.predict(X_blind.iloc[:, selected_features]),
            model.decision_function(X_blind.iloc[:, selected_features]),
        ),
    })
    print(f"completed RFE reference seed {seed}")

rfe_results = pd.DataFrame(rfe_rows)
rfe_path = repo_root / "validation/Synthetic100_sklearn_rfe_results.csv"
rfe_results.to_csv(rfe_path, index=False)
rfe_results

completed RFE reference seed 0


completed RFE reference seed 1


completed RFE reference seed 2


,seed,method,selection_strategy,num_models,num_unified_features,signal_recall,noise_fraction,member_disagreement,roc_auc,f1,balanced_accuracy,accuracy
0,0,sklearn SVM + RFE,RFE,1,20,0.4,0.6,0.0,0.875259,0.787402,0.794255,0.784
1,1,sklearn SVM + RFE,RFE,1,20,0.4,0.6,0.0,0.894151,0.774194,0.783644,0.776
2,2,sklearn SVM + RFE,RFE,1,20,0.4,0.6,0.0,0.894151,0.774194,0.783644,0.776


In [5]:
rfe_results.agg({
    "roc_auc": ["mean", "std"],
    "f1": ["mean", "std"],
    "balanced_accuracy": ["mean", "std"],
    "accuracy": ["mean", "std"],
    "signal_recall": ["mean", "std"],
    "noise_fraction": ["mean", "std"],
}).round(4)

,roc_auc,f1,balanced_accuracy,accuracy,signal_recall,noise_fraction
mean,0.8879,0.7786,0.7872,0.7787,0.4,0.6
std,0.0109,0.0076,0.0061,0.0046,0.0,0.0
